In [1]:
import os
if os.path.basename(os.getcwd()) == 'Demonstrations':
    os.chdir('..')

<div style="background-color: #daf1a1; color: white; padding: 30px; border-radius: 0px;">
<h1 style="margin: 0;  color: #002762">Visuals for the Paper on Stokes Equations </h3>
</div>

Imports:

In [2]:
from scipy.sparse.linalg import spsolve
import pandas as pd
import numpy as np

from Utilities.Stokes_felib import *
from Utilities.Mesh_processing import *

---

# Ladyzhenskaya–Babuška–Brezzi (LBB) Condition: Quick Summary

The LBB (inf-sup) condition is a necessary and sufficient stability criteria for saddle-point problems (like the Stokes equations) to ensure a unique, stable solution.

### 1. Continuous Definition
Velocity space $V$ and pressure space $Q$ must satisfy:
$$\exists \beta>0: \quad \inf_{q \in Q \setminus \{0\}} \space \sup_{v \in V \setminus \{0\}} \left(\frac{\int_{\Omega}q(\nabla \cdot v)d\Omega}{\|v\|_{V}\cdot\|q\|_{Q}}\right) \ge \beta$$

---

### 2. Discrete Algebraic System
Mapping continuous fields to nodal vectors ($\mathbf{u} \in \mathbb{R}^{2N_v}$, $\mathbf{p} \in \mathbb{R}^{N_p}$) yields the discrete inf-sup framework:

$$\exists \beta>0: \quad \inf_{\mathbf{p} \not= \mathbf{0}} \space \sup_{\mathbf{u} \not= \mathbf{0}} \left(\frac{\mathbf{p}^T \mathbf{B} \mathbf{u}}{\left(\mathbf{u}^T \mathbf{M}_v \mathbf{u}\right)^{1/2} \cdot \left(\mathbf{p}^T \mathbf{M}_p \mathbf{p}\right)^{1/2}}\right) \ge \beta$$

Where:
* **$\mathbf{B} = [\mathbf{B}_x \quad \mathbf{B}_y]$**: Discrete divergence matrix coupling pressure and velocity.
* **$\mathbf{M}_v = \begin{bmatrix} \mathbf{A} & \mathbf{0} \\ \mathbf{0} & \mathbf{A} \end{bmatrix}$**: Velocity energy weight matrix ($\mathbf{A}$ is the discrete Laplacian).
* **$\mathbf{M}_p$**: Pressure mass matrix accounting for spatial overlap of pressure basis functions.

---

### 3. Practical Stability Test (Generalized Eigenvalue Problem)
Maximizing over $\mathbf{u}$ and minimizing over $\mathbf{p}$ via the Rayleigh quotient transforms the condition into a straightforward matrix eigenvalue problem:

$$\exists \beta>0: \quad \sqrt{\lambda_{min}\left(\mathbf{M}_p^{-1}\left( \mathbf{B}_x \mathbf{A}^{-1} \mathbf{B}_x^T + \mathbf{B}_y \mathbf{A}^{-1} \mathbf{B}_y^T \right)\right)} \ge \beta$$

#### Grid Verification Rules:
* **Stable Grid Combinations:** As the mesh size $h \to 0$, the minimum eigenvalue $\lambda_{\min}$ bounds away from zero ($\lambda_{\min} \ge \beta^2 > 0$).
* **Unstable Grid Combinations:** As the mesh size $h \to 0$, $\lambda_{\min} \to 0$, indicating spurious pressure oscillations (checkerboard patterns).

---

### 4. Pressure Mass Matrix Assembly
To compute this stability bound numerically, the local pressure mass matrix over a triangular element with Jacobian determinant $|\mathbf{J}_T|$ is evaluated as:

$$\mathbf{M}_p^{loc} = \frac{\left|\mathbf{J}_T\right|}{24} \begin{bmatrix} 2 & 1 & 1\\ 1 & 2 & 1\\ 1 & 1 & 2 \end{bmatrix}$$

---